<b><font size="6" color="#E8800A">Week 2 · The Machine Learning process</font></b><br>
<b><font size="4">One complete pipeline, end to end: the map for the whole semester</font></b><br>

Last week you classified drug plantations **by hand** from labelled
examples. Today we hand exactly the same problem to Python and walk through
the **entire machine-learning pipeline** once, without stopping too long at
any stage: identify the business need, import and explore the data, prepare
it, build a model, **optimize** it, assess it, and **deploy** it on
new data.

Every stage gets its own deep-dive week later in the course. Today we see
all eight of them at once, none of them in depth.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>0 · Identify the business need</font>](#business)
* [<font color='#E8800A'>1 · Import the libraries</font>](#libraries)
* [<font color='#E8800A'>2 · Import the data</font>](#import-data)
* [<font color='#E8800A'>3 · Explore the data</font>](#explore)
* [<font color='#E8800A'>4 · Modify the data</font>](#modify)
* [<font color='#E8800A'>5 · Model: create a predictive model</font>](#model)
* [<font color='#E8800A'>6 · Optimize: tune the model</font>](#optimize)
* [<font color='#E8800A'>7 · Assess: how good is it really?</font>](#assess)
* [<font color='#E8800A'>8 · Deploy: use it on new data</font>](#deploy)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

![The machine-learning process: business need, import, explore, modify,
model, optimize, assess, deploy](../../assets/week_02/ml_process_pipeline.svg)

*The pipeline we build today. Each stage returns as a full week later in the
course.*

# <font color='#E8800A'>0 · Identify the business need</font> <a class="anchor" id="business"></a>
[Back to TOC](#toc)

First of all, we need to identify the business need: what decision does
this model support, and who acts on the answer?

**Our case:** the drug-plantation inspection agency. Field teams photographed
plantations; for 300 of them, inspectors confirmed the label
(`DrugPlant = 1` for drug plantations, `0` for legal crops). For 40 new
plantations we only have the four drone measurements `BD1` to `BD4`. The agency
wants a ranked list: **which new plantations should be inspected first?**

In the vocabulary of the five objects: one *observation* is a plantation; the *features*
are `BD1` to `BD4`; the *target* is `DrugPlant`; the *prediction moment* is
after the drone flight but before any inspection; the *decision* is where to
send the teams.

# <font color='#E8800A'>1 · Import the libraries</font> <a class="anchor" id="libraries"></a>
[Back to TOC](#toc)

The first step of every notebook is importing the tools:

- **pandas**: data manipulation and analysis;
- from **scikit-learn**, the library that will accompany us all semester:
  a `DecisionTreeClassifier` (our first model; how it works comes later in
  the course), `train_test_split`, and `confusion_matrix` and `f1_score`;
- **matplotlib** for one plot, and **joblib** for saving the final model.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import joblib

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score

RANDOM_STATE = 42
PLOT_BLUE, PLOT_ORANGE = "#1779A8", "#C96A00"  # course plot colours

# <font color='#E8800A'>2 · Import the data</font> <a class="anchor" id="import-data"></a>
[Back to TOC](#toc)

The data lives in an Excel file with two sheets, exactly how it left the
field team: `ClassifiedData` (300 labelled plantations) and `Data2Classify`
(40 new ones without labels). We import each sheet with pandas.

__Step 1:__ Import the sheet `ClassifiedData` from
`drug_plantations.xlsx` into an object named `plants_truth`, using
the column `ID` as index.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
# 1. read the workbook named in the step above: it sits in ../../data/raw/
plants_truth = pd.read_excel(
    ...,  # <-- CODE HERE
    # 2. the workbook holds more than one sheet, so name the one that
    #    carries the already-classified plantations
    sheet_name=...,  # <-- CODE HERE
    # 3. ID identifies a plantation, it is not a measurement: make it the
    #    index so it is not carried along as a feature
    index_col=...,  # <-- CODE HERE
)


__Step 2:__ Import the sheet `Data2Classify` into an object named
`plants_2classify` (same file, same index column).

In [ ]:
# 1. same workbook as the previous step
# 2. this sheet holds the rows that carry no label yet
# 3. index on the same column, so the ID does not arrive as a feature column
plants_2classify = pd.read_excel(
    ...,  # <-- CODE HERE
    sheet_name=...,  # <-- CODE HERE
    index_col=...,  # <-- CODE HERE
)


# <font color='#E8800A'>3 · Explore the data</font> <a class="anchor" id="explore"></a>
[Back to TOC](#toc)

Before modelling anything, look at the data. Exploration tells us what we
have, what is broken, and which features might carry signal. Exploration,
cleaning and feature work each get a deeper treatment later in the course.

__Step 3:__ Check the first five rows of `plants_truth` with `.head()`.

In [ ]:
# TODO: show the first five rows of plants_truth.


__Step 4:__ Use `.info()` to check the data types and whether any values are
missing.

In [ ]:
# TODO: check dtypes and missing values with .info().


__Step 5:__ Get the main descriptive statistics with `.describe()`.

In [ ]:
# TODO: descriptive statistics with .describe().


__Step 6:__ How many plantations of each class do we have? Use
`.value_counts()` on the target column.

In [ ]:
# TODO: count observations per class with .value_counts().


Note that the classes are **imbalanced**: legal crops dominate,
taking about 73 % of the rows. That number has two consequences for the rest of
the notebook.

**The split.** With one class this much rarer, a random split can easily hand the
two parts different class mixes, and a comparison between them would then reflect
the split as much as the model. `stratify=` in Step 9 prevents that.

**The metric.** A model that always answers "legal" scores 73 % accuracy while
catching zero drug plantations. Accuracy cannot tell that model apart from a
useful one. We still use accuracy in Stage 6, and Step 14 measures what it
hides.

__Step 7:__ Do the measurements differ between classes? Check the mean of each
feature per class with `.groupby("DrugPlant").mean()`.

In [ ]:
# TODO: mean of every feature per target class using groupby.


Which features would *you* rely on, looking at those group means? Write
your answer, then check whether the model agrees with you at the end.

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>4 · Modify the data</font> <a class="anchor" id="modify"></a>
[Back to TOC](#toc)

After exploring, we would normally fix problems (missing values,
outliers) and engineer better features; this dataset is clean, so today we
only do the two modifications every pipeline needs:

1. separate the **features** (`X`) from the **target** (`y`);
2. split the observations into a **training** part (to build the model) and
   a **validation** part (to judge it).

__Step 8:__ Create `X` with all columns except `DrugPlant`, and `y` with `DrugPlant` only.

In [ ]:
# TODO: create X (all feature columns) and y (the DrugPlant column).


<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html'>sklearn.model_selection.train_test_split(*arrays, train_size=None, shuffle=True, stratify=None, random_state=None)</a>

**Definition:**
Split arrays or matrices into random train and test subsets.

**Main Parameters:**
- `train_size` / `test_size`: proportion of observations in each part
- `shuffle`: whether to shuffle before splitting (default True)
- `stratify`: keep these class proportions equal in both parts
- `random_state`: makes the random split reproducible

**Main Attributes (after fitting):**
- *(function: returns the split arrays, keeps no state)*

**Main Methods:**
- *(call it directly: `X_train, X_val, y_train, y_val = train_test_split(X, y, ...)`)*
</div>

__Step 9:__ Split `X` and `y` into training (70%) and validation parts with
`train_test_split`, using `stratify=y` (for the reason measured in Step 6) and
`random_state=RANDOM_STATE` so the split is reproducible. (One split is not
enough for serious evaluation; that is taken up later in the course.)

In [ ]:
# 1. split X and y into a training part and a validation part
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    train_size=...,  # <-- CODE HERE
    # 2. shuffle, and hold the class balance of y in both halves,
    #    an unstratified split can starve one half of the rare class
    shuffle=...,  # <-- CODE HERE
    stratify=...,  # <-- CODE HERE
    # 3. fix the seed so a rerun puts the same rows in the same half
    random_state=...,  # <-- CODE HERE
)
# 4. report the two sizes
len(X_train), len(X_val)


<div class="alert alert-block alert-warning">

**The golden rule starts now.** From this line on, the validation rows
exist to *imitate the future*. Every decision (cleaning statistics,
feature choices, model settings) may look only at the **training** rows.
Break this rule and your assessment stops being an estimate of the future.
The name for breaking it is *leakage*.

</div>

# <font color='#E8800A'>5 · Model: create a predictive model</font> <a class="anchor" id="model"></a>
[Back to TOC](#toc)

Time to create a model. We use a **Decision Tree**, a model that
learns a cascade of if-then questions from the data, close in spirit to a
hand-written rule. How trees choose their questions comes later in the course;
today we only call `.fit()`.

**Why a tree, in a week that has taught no data preparation?** Because a tree
asks questions of the form "is `BD1` above this threshold?", and the answer does
not change if you rescale the column. It needs no scaling and no encoding, so it
can run on the raw measurements. Most other models need their inputs prepared
first.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html'>sklearn.tree.DecisionTreeClassifier(criterion='gini', max_depth=None, min_samples_split=2, random_state=None, ...)</a>

**Definition:**
A decision tree classifier: learns a hierarchy of feature thresholds that separates the classes.

**Main Parameters:**
- `max_depth`: maximum number of question levels (None = grow until pure)
- `criterion`: how splits are scored ('gini' or 'entropy')
- `min_samples_split`: minimum observations a node needs to split again
- `random_state`: makes tie-breaking reproducible

**Main Attributes (after fitting):**
- `classes_`: the class labels the tree knows
- `feature_importances_`: how much each feature contributed
- `get_depth()` / `get_n_leaves()`: the fitted tree's actual size

**Main Methods:**
- `fit(X, y)`: learn the tree from training data
- `predict(X)`: predict class labels
- `predict_proba(X)`: class probabilities (used for ranking!)
- `score(X, y)`: mean accuracy on the given data
</div>

__Step 10:__ Create a `DecisionTreeClassifier` named `tree` (pass
`random_state=RANDOM_STATE`) and fit it to the **training** data.

In [ ]:
# TODO: create the DecisionTreeClassifier (random_state=RANDOM_STATE)
# and fit it on X_train, y_train.


__Step 11:__ Predict the target for the training rows and for the validation rows
(`predictions_train`, `predictions_val`).

In [ ]:
# TODO: predict on X_train and on X_val, storing the results in
# predictions_train and predictions_val.


__Step 12:__ Before tuning anything, score the tree you just fitted. Take accuracy
with `.score()` and F1 with `f1_score`, on the training rows and on the
validation rows, so the default tree has four numbers of its own to be
compared against later.

In [ ]:
# TODO: accuracy with .score() and F1 with f1_score, for the default
# `tree`, on the training rows and on the validation rows. You already have
# predictions_train and predictions_val from Step 11.


A perfect 1.000 on the training rows against 0.833 on the validation
rows: the default tree memorised the training data. F1 says the same thing more
sharply, 1.000 against 0.694, because the rare class is where memorising shows
first.

Those four numbers are the baseline the Optimize stage has to beat.

# <font color='#E8800A'>6 · Optimize: tune the model</font> <a class="anchor" id="optimize"></a>
[Back to TOC](#toc)

A `DecisionTreeClassifier` grown without limits memorises the training
data. Most models have settings, **hyperparameters**, that control how
flexible they are; choosing them well is called *optimization* or *tuning*,
and it gets its own treatment later in the course.

Today we do the minimum version: try several values of `max_depth`, judge each
on the **validation** rows (never on training rows!), and keep the best.

**We judge on accuracy, which Step 6 already flagged as the wrong metric here.**
Defining a better one takes a week of its own, so we use accuracy today and
measure what it costs us before the stage is over.

In [ ]:
# One tree per candidate depth. None means "grow until the leaves
# are pure", so it is the most flexible option in the list.
depth_results = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    candidate = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE,
    )
    candidate.fit(X_train, y_train)
    # Train and validation for both metrics. The gap between the two columns
    # of a metric is what shows overfitting, so neither half is droppable.
    # F1 itself is defined properly later in the course.
    depth_results.append(
        {
            "max_depth": str(depth),
            "train_accuracy": candidate.score(X_train, y_train),
            "val_accuracy": candidate.score(X_val, y_val),
            "train_f1": f1_score(y_train, candidate.predict(X_train)),
            "val_f1": f1_score(y_val, candidate.predict(X_val)),
        }
    )

depth_table = pd.DataFrame(depth_results)
depth_table.round(3)

In [ ]:
# Two lines on one axis: where they separate is where the tree has
# started memorising rather than learning.
#
# One row per (depth, split) rather than two columns side by side. That is the
# shape a plotting library can read: the thing that differs between the lines
# becomes a column, and the library draws one line per value of it.
curves = depth_table.melt(
    id_vars="max_depth",
    value_vars=["train_accuracy", "val_accuracy"],
    var_name="measured on",
    value_name="accuracy",
).replace({"train_accuracy": "training", "val_accuracy": "validation"})

fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(curves, x="max_depth", y="accuracy", hue="measured on",
             marker="o", palette=[PLOT_BLUE, PLOT_ORANGE], ax=ax)
ax.set(xlabel="max_depth", ylabel="accuracy",
       title="Deeper trees memorise training data")
plt.show()

In [ ]:
# What does the metric choice cost? Look at the top of the ranking.
print("Best two by validation accuracy:")
print(
    depth_table.nlargest(2, "val_accuracy")[
        ["max_depth", "train_accuracy", "val_accuracy", "train_f1", "val_f1"]
    ].round(3).to_string(index=False)
)
print(f"\nalways-predict-legal accuracy: {1 - y_val.mean():.3f}")

**The top two are tied on accuracy.** Two different trees, the same
0.878, and nothing in that column to separate them, so the winner is whichever
row the sort put first.

Their `val_f1` values are not tied: 0.784 at depth 4 against 0.756 at depth 5. F1
weighs performance on the rare class, so it separates the two trees that accuracy
could not, and it prefers the shallower one.

The last line gives the floor: answering "legal" for every plantation scores 0.73
without inspecting anything. Any accuracy in this notebook has to be read against
that floor.

So accuracy costs us a decision here. We keep it today because the alternatives
are not defined yet, and we settle the tie in the next step by preferring the
smaller depth.

__Step 13:__ Read the best `max_depth` from the table (highest validation
accuracy) and refit the tree with it, on the training rows, as
`tree_tuned`. Break ties towards the **smaller** depth: when two trees score
the same, prefer the simpler one.

In [ ]:
ranked = depth_table.copy()
# 1. `max_depth` is held as text, and "None" means no depth limit, build a
#    numeric `depth_for_sorting` column in which "None" compares as deeper
#    than any numbered depth
ranked["depth_for_sorting"] = ...  # <-- CODE HERE
# 2. rank on validation score first, then on that numeric depth
#    (a tie settled by row order can hand the win to the deeper tree,
#     which costs complexity and returns no extra score)
best_row = ranked.sort_values(
    ...,  # <-- CODE HERE
    ascending=...,  # <-- CODE HERE
).iloc[0]
# 3. read the winning depth back out of `max_depth`: None when the text says
#    so, int otherwise
best_depth = ...  # <-- CODE HERE

# 4. rebuild the tree at that depth, seed fixed
tree_tuned = DecisionTreeClassifier(
    max_depth=...,  # <-- CODE HERE
    random_state=RANDOM_STATE,
)
# 5. fit it on the training rows
...  # <-- CODE HERE
best_depth, round(float(best_row["val_accuracy"]), 3)


# <font color='#E8800A'>7 · Assess: how good is it really?</font> <a class="anchor" id="assess"></a>
[Back to TOC](#toc)

We already peeked at accuracies while tuning; now look at the chosen
model on both parts of the data.

**One caveat first.** The validation rows chose `max_depth` in Stage 6, so they
are no longer neutral about it: we picked the winner partly on the luck of those
90 rows, which makes the winning score optimistic. It is still good enough for
*comparing* the candidates, but treat it as an upper bound on how this tree will
do on next month's drone batch. The fix is a third split that nothing is allowed
to choose on.

__Step 14:__ Check `tree_tuned` the same way you checked the default tree in
Step 12: accuracy with `.score()` and F1 with `f1_score`, on the training rows
and on the validation rows.

In [ ]:
# TODO: accuracy and F1 for tree_tuned, on the training rows and on the
# validation rows. Same four numbers as Step 12, so they can be compared.


If the training score is much higher than the validation score, the model
is **overfitting**: it memorised the training rows instead of learning the
pattern. Put these four numbers beside the four from Step 12: accuracy went from
1.000/0.833 to 0.962/0.878, and F1 from 1.000/0.694 to 0.929/0.784. The tuned
tree gave up training accuracy and the gap closed on both metrics.

Read the validation pair together as well. Accuracy is 0.878 against the 0.733
floor from Step 6, so the tree buys 0.145 over answering "legal" every time. F1
is 0.784, computed on the drug plantations alone, which is the class the agency
is paying to find. Report both.

Now put the two trees side by side on the validation rows.

In [ ]:
# Both trees on the validation rows, using the Step 12 baseline.
print(f"Default tree (unlimited depth): accuracy={untuned_val_accuracy:.3f}  F1={untuned_val_f1:.3f}")
print(f"Tuned tree   (max_depth={best_depth}):          accuracy={val_accuracy:.3f}  F1={val_f1:.3f}")
print(
    f"Change from tuning: accuracy {val_accuracy - untuned_val_accuracy:+.3f}"
    f"  F1 {val_f1 - untuned_val_f1:+.3f}"
)

Tuning traded a little training accuracy for a **better** validation
score. That is the Optimize stage doing its job.

It does not always work out that way. On a single validation split of 90 rows a
tuned model can come out level with the untuned default, or below it: the search
may have found nothing real, or the difference may be noise on a small split.
Report that result when you get it. It is one more reason to stop relying on a
single split, and every model family in this course has its own version of the
problem.

__Step 15:__ Accuracy hides *which* mistakes we make. Check the confusion matrix
on the validation rows:

```
[[TN, FP],
 [FN, TP]]
```

In [ ]:
# TODO: confusion matrix of tree_tuned on the validation rows
# (true labels first, predictions second).


Can we conclude something from the matrix? With imbalanced classes, the
model is usually better at the majority class, and the cost of a **false
negative** (missing a drug plantation) is not the same as a false positive
(a wasted inspection). Accuracy alone cannot express that; the F1 from Step 14
reads the matrix from the drug-plantation side. Precision, recall and the rest of
the family get their own week later in the course.

Look at the hand rule you wrote before class and at `tree_tuned.feature_importances_`
(run it in a scratch cell if you like). Did the model rely on the features
you predicted in Step 7? What does that tell you about exploration?

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>8 · Deploy: use it on new data</font> <a class="anchor" id="deploy"></a>
[Back to TOC](#toc)

The last stage is putting the model to work. "Deployment" can be
as heavy as a web service or as light as what the agency needs
today: score the 40 new plantations, export the ranked list, and **save the
model** so next month's drone batch can be scored without retraining.

__Step 16:__ Look at the new, unlabelled data (`plants_2classify`).

In [ ]:
# TODO: inspect the first rows of plants_2classify.


__Step 17:__ The agency wants a *ranked* list, so use `predict_proba` to get the
probability of class 1 for each new plantation, store it in a column named
`drug_probability`, and sort by it (highest first).

In [ ]:
# 1. work on a copy so plants_2classify stays as it was loaded
scored = plants_2classify.copy()
# 2. predict_proba returns one column per class, ordered as
#    tree_tuned.classes_, take the column belonging to class 1
scored["drug_probability"] = ...  # <-- CODE HERE
# 3. reorder the rows so the highest probability comes first
scored = ...  # <-- CODE HERE
scored.head(10)


__Step 18:__ Export the ranked list for the inspection teams as a CSV file.

In [ ]:
scored.to_csv("week_02_inspection_ranking.csv")
print("Exported", len(scored), "ranked plantations.")

__Step 19:__ Save the fitted model with `joblib`, then (pretending we are next
month's scoring script) load it back and verify it still predicts.

In [ ]:
joblib.dump(tree_tuned, "week_02_drug_plant_model.joblib")

reloaded_model = joblib.load("week_02_drug_plant_model.joblib")
reused = reloaded_model.predict(plants_2classify)
print("Reloaded model scored", len(reused), "new plantations.")

<div class="alert alert-block alert-success">

**Milestone!** You built the complete pipeline: business need → data →
exploration → preparation → model → optimization → assessment →
deployment on new data, with the model persisted for reuse. Every remaining
week of this course deepens exactly one part of what you just did.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

What this session established:

1. **A machine-learning project is a pipeline of eight stages.** Business need,
   data, exploration, preparation, model, tuning, assessment, deployment.
   Fitting the model is one stage of the eight, and it is seldom the stage that
   decides whether the work turns out to be useful.
2. **The data is prepared before the model exists, and the preparation is a
   record of decisions.** Every fill, drop and encoding is a choice you have to
   be able to defend afterwards.
3. **A tuning choice is judged on rows the model did not learn from.** The
   frame is split before anything is fitted, and the score you report comes from
   rows that took no part in choosing the winner.
4. **The metric you report decides what "good" means.** Accuracy 0.878 and F1
   0.784 describe the same tree, and the second one answers the agency's
   question.
5. **Deployment needs the model written to disk.** Scoring new, unlabelled rows
   is the deliverable, and `joblib` is what lets next month's batch reuse
   today's fit.

### Take it to your project

Add to your project notes:

1. Which pipeline stage do you expect to be **hardest** for your project
   problem, and why?
2. What would "deployment" concretely mean for it (a report? a ranked list?
   a saved model scoring new rows monthly?).

The semester project is graded on the whole pipeline, not only the model.

Exit ticket: why did we judge `max_depth` on the validation rows instead
of the training rows, and what would have happened if we had tuned on
training accuracy?

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- pandas, [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- scikit-learn, [Getting started](https://scikit-learn.org/stable/getting_started.html)
- scikit-learn, [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- joblib, [Persistence](https://joblib.readthedocs.io/en/latest/persistence.html)